In [ ]:
import pandas as pd

# Charger le fichier parquet
df = pd.read_parquet("../data/train-00000-of-00001-5fae92537ff6cc9b.parquet")

print("Nombre total de lignes :", len(df))
print("\nColonnes :", df.columns.tolist())

print("\n=== Distribution de Irony_Labels ===")
print(df["Irony_Labels"].value_counts())
print("\n", df["Irony_Labels"].value_counts(normalize=True))

In [ ]:
#Créer le label binaire

In [ ]:
# Créer la colonne binaire : ironique (sarcastic + humourus) = 1, not_ironic = 0
df["label"] = df["Irony_Labels"].apply(lambda x: 0 if x == "not_ironic" else 1)

print("=== Distribution du nouveau label binaire ===")
print(df["label"].value_counts())
print("\n", df["label"].value_counts(normalize=True))

In [ ]:
#Split train/test stratifié

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df["label"], 
    random_state=42
)

print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")

print("\n=== Distribution Train ===")
print(df_train["label"].value_counts(normalize=True))

print("\n=== Distribution Test ===")
print(df_test["label"].value_counts(normalize=True))

# Sauvegarder
df_train.to_csv("../data/train.csv", index=False)
df_test.to_csv("../data/test.csv", index=False)

print("\n✅ train.csv et test.csv sauvegardés !")

In [ ]:
# Nettoyage du texte

In [ ]:
import re
import pyarabic.araby as ar
import emoji

def clean_tweet_with_emoji(tweet):
    if not isinstance(tweet, str):
        return ""
    # Suppression des URLs
    tweet = re.sub(r'http\S+|www.\S+', '', tweet)
    # Suppression des mentions et hashtags
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    # Garder uniquement : arabe + espaces + emojis
    tweet = ''.join(c for c in tweet if 
                     ('\u0600' <= c <= '\u06FF') or  # arabe
                     c.isspace() or 
                     c in emoji.EMOJI_DATA)            # emojis
    # Normalisation des caractères arabes
    tweet = ar.normalize_hamza(tweet)
    tweet = ar.normalize_ligature(tweet)
    tweet = ar.strip_tashkeel(tweet)
    return tweet.strip()

df_train["cleaned_tweet"] = df_train["Emoji_Text"].apply(clean_tweet_with_emoji)
df_test["cleaned_tweet"] = df_test["Emoji_Text"].apply(clean_tweet_with_emoji)

print("✅ Nettoyage terminé !")
print(df_train[["Emoji_Text", "cleaned_tweet"]].head())